In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")

print("Path to dataset files:", path)

Path to dataset files: /Users/diyagajjar/.cache/kagglehub/datasets/rohanrao/formula-1-world-championship-1950-2020/versions/24


In [2]:
import os
print(os.listdir(path))

['circuits.csv', 'status.csv', 'lap_times.csv', 'sprint_results.csv', 'drivers.csv', 'races.csv', 'constructors.csv', 'constructor_standings.csv', 'qualifying.csv', 'driver_standings.csv', 'constructor_results.csv', 'pit_stops.csv', 'seasons.csv', 'results.csv']


In [3]:
import pandas as pd

# Load Kaggle Files
qualifying = pd.read_csv('qualifying.csv')
races = pd.read_csv('races.csv')
circuits = pd.read_csv('circuits.csv')
drivers = pd.read_csv('drivers.csv')

# Merge to get the full story (Names, Years, Circuits)
df = qualifying.merge(races[['raceId', 'year', 'name', 'circuitId']], on='raceId')
df = df.merge(circuits[['circuitId', 'name', 'location']], on='circuitId', suffixes=('_gp', '_circuit'))
df = df.merge(drivers[['driverId', 'surname']], on='driverId')

# Clean Q1 Times (Focusing on Q1 for the 107% rule)
def time_to_sec(t):
    try:
        if ":" in str(t):
            m, s = str(t).split(":")
            return int(m) * 60 + float(s)
        return float(t)
    except: return None

df['q1_sec'] = df['q1'].apply(time_to_sec)
df = df.dropna(subset=['q1_sec'])

# CALCULATE: 107% Threshold and Engine Eras
df['pole_q1'] = df.groupby('raceId')['q1_sec'].transform('min')
df['pct_of_pole'] = (df['q1_sec'] / df['pole_q1']) * 100
df['threshold_107'] = df['pole_q1'] * 1.07
df['is_safe_107'] = df['q1_sec'] <= df['threshold_107']

def get_era(year):
    if year <= 2013: return 'V8 Era'
    elif year <= 2021: return 'Hybrid Era'
    else: return 'Ground Effect'
df['engine_era'] = df['year'].apply(get_era)

# 1. THE DEFINITIVE STREET LIST (2009-2024)
street_circuits = [
    'Albert Park Grand Prix Circuit',      # Melbourne
    'Baku City Circuit',                   # Baku
    'Circuit de Monaco',                   # Monaco
    'Circuit Gilles Villeneuve',           # Montreal
    'Jeddah Corniche Circuit',             # Jeddah
    'Las Vegas Strip Street Circuit',      # Vegas
    'Marina Bay Street Circuit',           # Singapore
    'Miami International Autodrome',       # Miami
    'Sochi Autodrom',                      # Sochi (Olympic Park Street Circuit)
    'Valencia Street Circuit'              # Valencia (Old European GP)
]

# 2. APPLY TO YOUR DATAFRAME
df['track_type'] = 'Permanent'
df.loc[df['name_circuit'].isin(street_circuits), 'track_type'] = 'Street'

# 3. VERIFICATION COUNT
# This should return exactly 10 unique names for Street
print(f"Street tracks found: {df[df['track_type'] == 'Street']['name_circuit'].nunique()}")

# FILTER: Keep only 2009 onwards (15-year window)
df = df[df['year'] >= 2009]

# EXPORT: Only the necessary variables
final_cols = [
    'year', 'engine_era', 'name_gp', 'name_circuit', 
    'surname', 'track_type', 'q1_sec', 'pct_of_pole', 'is_safe_107']

df[final_cols].to_csv('f1_historical.csv', index=False)
print(f"SUCCESS: Created with {len(df)} rows from 2009-2024.")

Street tracks found: 10
SUCCESS: Created with 6689 rows from 2009-2024.


In [4]:
print(df[final_cols].head())

      year engine_era                name_gp                    name_circuit  \
2984  2009     V8 Era  Australian Grand Prix  Albert Park Grand Prix Circuit   
2985  2009     V8 Era  Australian Grand Prix  Albert Park Grand Prix Circuit   
2986  2009     V8 Era  Australian Grand Prix  Albert Park Grand Prix Circuit   
2987  2009     V8 Era  Australian Grand Prix  Albert Park Grand Prix Circuit   
2988  2009     V8 Era  Australian Grand Prix  Albert Park Grand Prix Circuit   

          surname track_type  q1_sec  pct_of_pole  is_safe_107  
2984       Button     Street  85.211   100.241159         True  
2985  Barrichello     Street  85.006   100.000000         True  
2986       Vettel     Street  85.938   101.096393         True  
2987       Kubica     Street  85.922   101.077571         True  
2988      Rosberg     Street  85.846   100.988166         True  


In [24]:
import requests
import pandas as pd
import time
from datetime import timedelta

# 1. SETTINGS
SESSION_KEY = 9506 
DRIVERS = {'Verstappen': 1, 'Perez': 11}
all_telemetry = []

print("Starting telemetry fetch...")

for name, d_num in DRIVERS.items():
    print(f"Fetching {name} (# {d_num})...")
    
    # A. Find the fastest lap window
    laps_res = requests.get(f"https://api.openf1.org/v1/laps?session_key={SESSION_KEY}&driver_number={d_num}")
    laps = laps_res.json()
    
    # Sort to find the fastest lap duration
    fastest_lap = sorted([l for l in laps if l.get('lap_duration')], key=lambda x: x['lap_duration'])[0]
    
    # B. Define the time window for the sensors
    t1 = pd.to_datetime(fastest_lap['date_start'])
    t2 = t1 + timedelta(seconds=fastest_lap['lap_duration'])
    
    # C. Pull the sensor data
    tel_url = f"https://api.openf1.org/v1/car_data?session_key={SESSION_KEY}&driver_number={d_num}&date>={t1.isoformat()}&date<={t2.isoformat()}"
    tel_res = requests.get(tel_url).json()
    
    if tel_res:
        df = pd.DataFrame(tel_res)
        
        # --- ADD THE TEAM NAME HERE ---
        df['driver_name'] = name
        df['team'] = 'Red Bull Racing'  # Explicitly naming the team for your report
        
        # Tagging for Observable logic
        df['performance_status'] = 'Benchmark' if name == 'Verstappen' else 'Risk' 
        
        # Normalize Time
        df['date'] = pd.to_datetime(df['date'])
        df['time_offset'] = (df['date'] - df['date'].min()).dt.total_seconds()
        
        all_telemetry.append(df)
    
    time.sleep(0.5)

# 2. SUBSET & EXPORT
necessary_cols = [
    'time_offset', 'driver_name', 'performance_status', 
    'throttle', 'brake', 'speed', 'rpm', 'n_gear'
]

final_telemetry = pd.concat(all_telemetry)[necessary_cols]
final_telemetry.to_csv('f1_telemetry.csv', index=False)

print("\nSUCCESS: f1_telemetry.csv is ready!")

Starting telemetry fetch...
Fetching Verstappen (# 1)...
Fetching Perez (# 11)...

SUCCESS: f1_telemetry.csv is ready!
